# 🧪 Parameter Selection & Data Standardization

**Author:** Gabriella Marín  
**Project:** Multi-Layer Water Quality Risk & Regulatory Analytics System
**Phase:** Phase 1 – Data Preparation

**Objective:**  
Clean and standardize the core monitoring parameters by normalizing parameter names and converting raw
measurement values into numeric form while explicitly flagging censored results (e.g., "<2").
Then, reshape the standardized long-format dataset into a sample-level wide table (one row per sample)
to serve as the foundation for SQL-based normative compliance and risk scoring.

## Load data, column fefinitions and core filtering

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load dataset
PROJECT_DIR = Path(r"D:\Documents\Portfolio\01-water-quality-normative")
DATA_FILE = PROJECT_DIR / "datos_calidad_del_agua_2005_2024.xlsx"

df_raw = pd.read_excel(DATA_FILE,sheet_name="BASE DE DATOS",header=5)

# Paths
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Key columns (identified in Notebook 1)
DATE_COL = "FECHA"
PARAM_COL = "PROPIEDAD OBSERVADA "
VALUE_COL = "*RESULTADO"
UNIT_COL  = "UNIDAD DEL RESULTADO"

SAMPLE_ID_COL = "CODIGO__MUESTRA"  # unique sample identifier in the dataset

# Core parameters (defined in Phase 0)
# NOTE: Chloride may be excluded later due to low coverage; keep for traceability.
CORE_PARAMETERS = [
    "pH",
    "TURBIDEZ",
    "CONDUCTIVIDAD ELECTRICA",
    "SOLIDOS SUSPENDIDOS TOTALES",
    "TEMPERATURA",
    "OXIGENO DISUELTO (OD)",
    "DEMANDA BIOQUIMICA DE OXIGENO (DBO5)",
    "DEMANDA QUIMICA DE OXIGENO (DQO)",
    "NITRATO",
    "NITRITO",
    "NITROGENO AMONIACAL",
    "SULFATO",
    "CLORURO"]

# Filter dataset to core parameters
df_core = df_raw[df_raw[PARAM_COL].isin(CORE_PARAMETERS)].copy()

print("Filtered dataset shape:", df_core.shape)
display(df_core[PARAM_COL].value_counts())

Filtered dataset shape: (67630, 17)


PROPIEDAD OBSERVADA 
CONDUCTIVIDAD ELECTRICA                 6771
pH                                      6764
TEMPERATURA                             6738
DEMANDA QUIMICA DE OXIGENO (DQO)        6694
SOLIDOS SUSPENDIDOS TOTALES             6660
OXIGENO DISUELTO (OD)                   6559
TURBIDEZ                                6549
NITRITO                                 5129
NITRATO                                 5101
NITROGENO AMONIACAL                     4997
SULFATO                                 4199
DEMANDA BIOQUIMICA DE OXIGENO (DBO5)    1461
CLORURO                                    8
Name: count, dtype: int64

## 7. Parameter Name Normalization and Value Cleaning

In [12]:
#Parameter name standardization
PARAMETER_MAP = {
    "pH": "pH",
    "TURBIDEZ": "Turbidity",
    "CONDUCTIVIDAD ELECTRICA": "Electrical Conductivity",
    "SOLIDOS SUSPENDIDOS TOTALES": "Total Suspended Solids",
    "TEMPERATURA": "Temperature",
    "OXIGENO DISUELTO (OD)": "Dissolved Oxygen",
    "DEMANDA BIOQUIMICA DE OXIGENO (DBO5)": "BOD5",
    "DEMANDA QUIMICA DE OXIGENO (DQO)": "COD",
    "NITRATO": "Nitrate",
    "NITRITO": "Nitrite",
    "NITROGENO AMONIACAL": "Ammoniacal Nitrogen",
    "SULFATO": "Sulfate",
    "CLORURO": "Chloride"}

df_core["parameter_std"] = df_core[PARAM_COL].map(PARAMETER_MAP)

# Value cleaning + censoring flags
df_core["value_raw"] = df_core[VALUE_COL].astype(str).str.strip()
df_core["is_censored"] = df_core["value_raw"].str.startswith(("<", ">"))

clean = (df_core["value_raw"]
        .str.replace("<", "", regex=False)
        .str.replace(">", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip())

df_core["value_num"] = pd.to_numeric(clean, errors="coerce")

# Unit preservation
df_core["unit_raw"] = df_core[UNIT_COL].astype(str)
display(df_core[[PARAM_COL, "parameter_std", "value_raw", "is_censored", "value_num", "unit_raw"]].head(10))

# Unit consistency check (documentation)
unit_check = df_core.groupby("parameter_std")["unit_raw"].unique().reset_index()
display(unit_check)

print("Core long-format shape:", df_core.shape)

,PROPIEDAD OBSERVADA,parameter_std,value_raw,is_censored,value_num,unit_raw
26,CONDUCTIVIDAD ELECTRICA,Electrical Conductivity,60,False,60.00,µS/cm
27,CONDUCTIVIDAD ELECTRICA,Electrical Conductivity,85.7,False,85.70,µS/cm
28,CONDUCTIVIDAD ELECTRICA,Electrical Conductivity,109.2,False,109.20,µS/cm
29,CONDUCTIVIDAD ELECTRICA,Electrical Conductivity,92.18,False,92.18,µS/cm
38,DEMANDA BIOQUIMICA DE OXIGENO (DBO5),BOD5,2.6,False,2.60,mg O2/L
39,DEMANDA BIOQUIMICA DE OXIGENO (DBO5),BOD5,<2,True,2.00,mg O2/L
40,DEMANDA BIOQUIMICA DE OXIGENO (DBO5),BOD5,<2,True,2.00,mg O2/L
41,DEMANDA BIOQUIMICA DE OXIGENO (DBO5),BOD5,5,False,5.00,mg O2/L
42,DEMANDA QUIMICA DE OXIGENO (DQO),COD,63,False,63.00,mg O2/L
43,DEMANDA QUIMICA DE OXIGENO (DQO),COD,11,False,11.00,mg O2/L


,parameter_std,unit_raw
0,Ammoniacal Nitrogen,[mg N-NH3-/L]
1,BOD5,[mg O2/L]
2,COD,[mg O2/L]
3,Chloride,[mg Cl-/L]
4,Dissolved Oxygen,[mg O2/L]
5,Electrical Conductivity,[µS/cm]
6,Nitrate,[mg N-NO3-/L]
7,Nitrite,"[mg N-NO2-/L, mg/L, mg NO2-N/L]"
8,Sulfate,[mg SO4-2/L]
9,Temperature,[°C]


Core long-format shape: (67630, 22)


## 8. Sample Key Definition

A "sample" is defined using the dataset's unique sample identifier (`CODIGO__MUESTRA`), which is linked
to monitoring location metadata and sampling date. Aggregation is performed at this sample level.

In [13]:
# Basic sanity checks for sample key
if SAMPLE_ID_COL not in df_core.columns:
    raise KeyError(f"Sample id column not found: {SAMPLE_ID_COL}")

# How many unique samples?
n_samples = df_core[SAMPLE_ID_COL].nunique()
print("Unique samples:", n_samples)

# Detect duplicates at (sample_id, parameter_std)
dup_counts = (df_core.groupby([SAMPLE_ID_COL, "parameter_std"])
            .size().reset_index(name="n_records"))

print("Duplicate pairs (sample, parameter) with n_records > 1:")
display(dup_counts[dup_counts["n_records"] > 1].head(20))

print("Share of duplicated (sample, parameter) pairs:",(dup_counts["n_records"] > 1).mean())

Unique samples: 6818
Duplicate pairs (sample, parameter) with n_records > 1:


,CODIGO__MUESTRA,parameter_std,n_records
51950,27431,Dissolved Oxygen,2
52084,27457,COD,2


Share of duplicated (sample, parameter) pairs: 2.9573549417401077e-05


## 9. Aggregation Rule

If multiple measurements exist for the same parameter within the same sample, values are aggregated using
the mean of the numeric values. Raw values, censoring flags, and original units remain available in the
long-format table for traceability.

In [14]:
# Keep sample metadata columns (minimal set for traceability)
META_COLS = [
    SAMPLE_ID_COL,
    DATE_COL,
    "NOMBRE DEL PUNTO DE MONITOREO",
    "DEPARTAMENTO",
    "MUNICIPIO",
    "LATITUD",
    "LONGITUD",]

# Ensure metadata columns exist (skip missing gracefully)
META_COLS = [c for c in META_COLS if c in df_core.columns]

# Aggregate numeric values per (sample_id, parameter_std)
df_agg = (df_core.groupby(META_COLS + ["parameter_std"], dropna=False)["value_num"]
            .mean().reset_index())

# Pivot to wide format
df_wide = df_agg.pivot_table(index=META_COLS,columns="parameter_std",
            values="value_num",aggfunc="mean").reset_index()

# Flatten columns if needed
df_wide.columns.name = None

print("Wide dataset shape:", df_wide.shape)
display(df_wide.head())

Wide dataset shape: (6854, 20)


,CODIGO__MUESTRA,FECHA,NOMBRE DEL PUNTO DE MONITOREO,DEPARTAMENTO,MUNICIPIO,LATITUD,LONGITUD,Ammoniacal Nitrogen,BOD5,COD,Chloride,Dissolved Oxygen,Electrical Conductivity,Nitrate,Nitrite,Sulfate,Temperature,Total Suspended Solids,Turbidity,pH
0,11284,2005-02-15,RCA_BOGOTA_CUN_VILLAPINZON_PTE.CARRETERA-BOGOTA,CUNDINAMARCA,VILLAPINZÓN,5.218611,-73.595556,0.0448,NaN,20.0,NaN,7.2,53.9,0.22,0.0060,3.0,13.2,4.5,5.0,6.96
1,11285,2005-02-15,RCA_BOGOTA_CUN_TOCANCIPA_PTE.TULIO BOTERO-BOGOTA,CUNDINAMARCA,TOCANCIPÁ,4.971917,-73.916139,NaN,2.0,33.0,NaN,6.2,52.2,NaN,NaN,NaN,17.6,31.0,25.0,6.78
2,11289,2005-02-15,RCA_BOGOTA_CUN_VILLAPINZON_SANPEDRO-BOGOTA,CUNDINAMARCA,VILLAPINZÓN,5.194722,-73.613889,4.0300,59.5,160.0,NaN,2.1,658.0,1.80,0.0085,60.0,14.5,44.0,28.0,10.08
3,11291,2005-02-16,RCA_BOGOTA_CUN_SOACHA_ALICACHIN - EL SALTO [21...,CUNDINAMARCA,SOACHA,4.544847,-74.258269,15.3000,76.8,190.0,NaN,0.0,621.0,0.69,0.0060,36.0,19.2,52.0,42.0,7.06
4,11292,2005-02-16,RCA_BOGOTA_CUN_SOACHA_ALICACHIN - EL SALTO [21...,CUNDINAMARCA,SOACHA,4.544847,-74.258269,14.3000,NaN,170.0,NaN,4.2,602.0,0.65,0.0060,28.0,19.0,80.0,33.0,7.76


## 10. Coverage diagnostics in wide table

In [15]:
# Parameter completeness at sample level
param_cols = [c for c in df_wide.columns if c not in META_COLS]

coverage = (df_wide[param_cols].notna().mean().sort_values(ascending=False)
            .mul(100).round(1))

print("Sample-level coverage (% non-null):")
display(coverage)

# Phase 1 outputs export (CSV fallback)
df_core.to_csv(OUTPUT_DIR / "phase1_clean_long.csv", index=False)
df_wide.to_csv(OUTPUT_DIR / "phase1_sample_wide.csv", index=False)

print("Phase 1 outputs exported successfully (CSV).")

Sample-level coverage (% non-null):


Electrical Conductivity    98.8
pH                         98.7
Temperature                98.3
COD                        97.7
Total Suspended Solids     97.2
Dissolved Oxygen           95.7
Turbidity                  95.6
Nitrite                    74.8
Nitrate                    74.4
Ammoniacal Nitrogen        72.9
Sulfate                    61.3
BOD5                       21.3
Chloride                    0.1
dtype: float64

Phase 1 outputs exported successfully (CSV).


## Summary

Core monitoring parameters were filtered and standardized into an analysis-ready long table,
preserving raw values, units, and explicit flags for censored measurements reported below detection limits.

Numeric conversion was applied without imputation to maintain traceability and avoid introducing
artificial patterns. The dataset was then reshaped into a sample-level wide table (`df_wide`),
where each row represents a unique sample and each standardized parameter becomes a feature column.

Coverage diagnostics indicate strong availability for key physicochemical and nutrient parameters,
while chloride shows insufficient coverage and may be excluded from downstream modeling.
Phase 1 deliverables were exported as CSV files for the next phases (SQL compliance and risk scoring).